# Part 3: GenAI Insight Narrator (Gemini + offline fallback)

This notebook runs the same code as **narrator/generate_narrative.py**, split into cells.

1. Run the install cell, then upload **findings.json**. Create it with the Part 2 notebook or with **python analysis/clean_and_eda.py**.
2. **Gemini key (optional):** in Colab, open **Secrets** (the key icon on the left) and add your free Google AI Studio key under the name **SY_API_KEY**. Turn on notebook access for it.
  
Without a key, the notebook uses the offline template and still produces the full narrative.

In [1]:
!pip install -q google-genai

In [2]:
# Upload findings.json and put it in a narrator/ folder (the code reads narrator/findings.json)
import os
os.makedirs("narrator", exist_ok=True)
try:
    from google.colab import files
    uploaded = files.upload()          # select findings.json
    for name in uploaded:
        os.replace(name, os.path.join("narrator", "findings.json"))
except ImportError:
    print("Not running in Colab - using narrator/findings.json from the current folder.")
print("narrator/findings.json found:", os.path.exists("narrator/findings.json"))

Saving findings.json to findings.json
narrator/findings.json found: True


In [3]:
# Read the Gemini API key from Colab Secrets into an environment variable.
# If there is no key, the narrator automatically uses the offline template.
import os
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("SY_API_KEY")   # the name of your Colab secret
    print("Gemini key loaded from Colab Secrets.")
except Exception:
    print("No Gemini key found - the offline template will be used.")

Gemini key loaded from Colab Secrets.


## Setup: imports and settings

In [4]:
import json
import os
import re
import sys
from datetime import datetime

try:
    from google import genai
    from google.genai import types, errors
except ImportError:
    genai = None

FINDINGS_PATH = "narrator/findings.json"
SAMPLE_OUTPUT_PATH = "narrator/sample_output.txt"
MODEL_NAME = "gemini-3.1-flash-lite"     # free-tier Gemini model

## Step 1: Load the verified numbers from Part 2

In [5]:
def load_findings(path=FINDINGS_PATH):
    """Read findings.json (written by analysis/clean_and_eda.py)."""
    if not os.path.exists(path):
        print(f"ERROR: {path} not found.")
        print("Run 'python analysis/clean_and_eda.py' first - its last step writes this file.")
        sys.exit(1)
    with open(path) as f:
        return json.load(f)


def month_name(year_month):
    """Turn '2026-03' into 'March 2026'."""
    return datetime.strptime(year_month, "%Y-%m").strftime("%B %Y")

## Step 2: Online path - generate the SCR narrative with Gemini

In [6]:
# The system instruction fixes the role, the structure and the "no invented numbers" rule.
# It is passed separately (system_instruction), not mixed into the user prompt.
SYSTEM_INSTRUCTION = """You are a senior data analyst writing for Mamaearth's regional ops and finance heads.

Write a short business narrative (about 250 words) with exactly three labeled sections, in this order:
Situation:
Complication:
Resolution:

Rules:
- Every number you write must come from the findings supplied by the user and must keep exactly the same value.
- Do not invent, estimate, round differently or calculate any new statistics.
- Write money in Indian Rupees with two decimals and comma separators, for example Rs 97,358.30.
- Write percentages with one decimal, for example 44.4%.
- Write months by name, for example March 2026.
- Plain text only: no markdown symbols, no bullet characters, no tables.
- The Resolution must give clear actions the ops and finance heads can take."""


def build_user_prompt(findings):
    """Build the user prompt from the findings dict. No number is typed in here:
    every value is taken from the argument, so a different findings.json gives a different prompt."""
    rates = findings["return_rate_by_payment"]
    segment = findings["highest_risk_segment"]
    peak = findings["true_peak_month"]
    inflated = findings["outlier_inflated_month"]
    extra = findings.get("extra_context", {})

    rate_lines = "\n".join(f"  - {method}: {rate}%" for method, rate in rates.items())

    prompt = f"""Write the SCR narrative using ONLY these verified findings.

Revenue:
  - Raw total revenue (all orders, including duplicates): Rs {findings['raw_total_revenue_inr']:,.2f}
  - Cleaned total revenue (after removing duplicates): Rs {findings['cleaned_total_revenue_inr']:,.2f}
  - Difference caused by duplicate (double-submitted) orders: Rs {findings['duplicate_reconciliation_delta_inr']:,.2f}

Return rate by payment method:
{rate_lines}

Highest-risk segment: {segment['payment_method']} orders in Tier-{segment['city_tier']} cities, return rate {segment['return_rate_pct']}%

Monthly revenue:
  - True peak month (outliers excluded): {month_name(peak['month'])}, Rs {peak['revenue_inr']:,.2f}
  - {month_name(inflated['month'])} looked like the best month at Rs {inflated['apparent_revenue_inr']:,.2f} only because of bulk outlier orders; corrected it is Rs {inflated['corrected_revenue_inr']:,.2f}

Extra context (all findings as JSON):
{json.dumps(extra, indent=2)}"""
    return prompt


def generate_scr_narrative(findings: dict) -> dict:
    """Online path: ask Gemini for the SCR narrative.
    Always returns a dict - the caller never receives a raw exception."""
    try:
        if genai is None:
            raise RuntimeError("google-genai is not installed (pip install google-genai)")

        api_key = os.environ.get("GEMINI_API_KEY")
        if not api_key:
            raise RuntimeError("No API key: the GEMINI_API_KEY environment variable is not set")

        client = genai.Client(api_key=api_key)

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=build_user_prompt(findings),
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_INSTRUCTION,
                # temperature 0.0 = as deterministic as possible. This is a factual business
                # report built from verified numbers, not creative writing.
                temperature=0.0,
                # Explicit output limit: enough for a 3-section, ~250-word narrative
                # Too low a limit can cut the narrative off part-way
                max_output_tokens=2048,
                # Timeout is in MILLISECONDS: 30000 = 30 seconds (Gemini's minimum is 10 seconds)
                http_options=types.HttpOptions(timeout=30000),
            ),
        )

        narrative = (response.text or "").strip()
        finish_reason = str(response.candidates[0].finish_reason)

        # Treat an empty or cut-off answer as a failure, so the offline fallback is used
        if not narrative:
            raise RuntimeError(f"Gemini returned no text (finish_reason: {finish_reason})")
        if "MAX_TOKENS" in finish_reason:
            raise RuntimeError("Gemini output was cut off by max_output_tokens")

        return {
            "status": "success",
            "narrative": narrative,
            "tokens": response.usage_metadata.total_token_count,
            "source": f"gemini ({MODEL_NAME})",
        }

    except Exception as err:
        # Any problem (no key, Gemini APIError, network error, timeout, cut-off output)
        # becomes a structured error dict instead of crashing the program.
        return {"status": "error", "narrative": None, "message": str(err)}

## Step 3: Offline path - the same SCR narrative from a template (no API, no key)

In [7]:
def generate_scr_narrative_offline(findings: dict) -> dict:
    """Deterministic fallback: fills an f-string template with the numbers from findings.
    No network call and no API key. Returns the same dict shape as the online path."""
    rates = findings["return_rate_by_payment"]
    segment = findings["highest_risk_segment"]
    peak = findings["true_peak_month"]
    inflated = findings["outlier_inflated_month"]
    extra = findings.get("extra_context", {})

    # Payment methods from highest to lowest return rate, e.g. "COD 44.4%, UPI 18.9%, CARD 14.7%"
    rate_text = ", ".join(f"{m} {r}%" for m, r in sorted(rates.items(), key=lambda x: -x[1]))
    worst_method = max(rates, key=rates.get)
    best_method = min(rates, key=rates.get)

    # Optional detail: Tier-1 rate for the same payment method (only if it was exported)
    tier1_text = ""
    if "cod_tier1_return_rate_pct" in extra and segment["payment_method"] == "COD":
        tier1_text = (f" In Tier-1 cities the same COD orders return at "
                      f"{extra['cod_tier1_return_rate_pct']}%, so the problem is concentrated, not uniform.")

    duplicate_ids = ", ".join(extra.get("duplicate_order_ids_removed", []))
    outlier_ids = " and ".join(extra.get("outlier_order_ids", []))

    narrative = f"""Situation:
After cleaning the order data, Mamaearth's verified total revenue is Rs {findings['cleaned_total_revenue_inr']:,.2f}. The raw order table showed Rs {findings['raw_total_revenue_inr']:,.2f}; the difference of Rs {findings['duplicate_reconciliation_delta_inr']:,.2f} comes entirely from double-submitted duplicate orders{f' ({duplicate_ids})' if duplicate_ids else ''}, which were removed.

Complication:
Returns are concentrated in one payment method. Return rates by payment method are {rate_text}, so {worst_method} orders come back far more often than {best_method} orders. The highest-risk segment is {segment['payment_method']} orders in Tier-{segment['city_tier']} cities, with a return rate of {segment['return_rate_pct']}%.{tier1_text} The monthly trend was also misleading: {month_name(inflated['month'])} looked like the best month at Rs {inflated['apparent_revenue_inr']:,.2f} only because of bulk outlier orders{f' ({outlier_ids})' if outlier_ids else ''}; without them it is Rs {inflated['corrected_revenue_inr']:,.2f}, and the true peak month is {month_name(peak['month'])} at Rs {peak['revenue_inr']:,.2f}.

Resolution:
Regional ops should target {segment['payment_method']} orders in Tier-{segment['city_tier']} cities first: confirm these orders before dispatch and offer an incentive to switch to prepaid payment. Finance should report revenue only after removing duplicate orders and should fix the double-submit bug at checkout. Bulk orders should be flagged and reviewed separately, and {month_name(peak['month'])}, not {month_name(inflated['month'])}, should be used as the true peak month for planning."""

    return {
        "status": "success",
        "narrative": narrative,
        "tokens": 0,                      # no API call, so no tokens used
        "source": "offline template",
    }

## Step 4: Numeric accuracy checker (the five required figures)

In [8]:
def number_forms(value):
    """Ways a number may be written, after commas are removed: 97358.3 -> {'97358.30', '97358.3'}."""
    return {f"{value:.2f}", f"{value:.1f}", f"{value:g}"}


def check_numeric_accuracy(narrative, findings):
    """Check that the narrative contains all five key figures. Prints PASS/FAIL per figure.
    The expected values come from findings.json, not typed in here."""
    text = narrative.replace(",", "")        # normalize: "97,358.30" -> "97358.30"
    peak = findings["true_peak_month"]
    peak_month_word = month_name(peak["month"]).split()[0]      # e.g. "March"

    checks = [
        ("Cleaned total revenue", findings["cleaned_total_revenue_inr"], None),
        ("COD return rate", findings["return_rate_by_payment"]["COD"], None),
        ("Highest-risk segment rate", findings["highest_risk_segment"]["return_rate_pct"], None),
        ("Duplicate reconciliation delta", findings["duplicate_reconciliation_delta_inr"], None),
        ("True peak month + revenue", peak["revenue_inr"], peak_month_word),
    ]

    all_passed = True
    for label, value, word in checks:
        found = any(form in text for form in number_forms(value))
        if word is not None:
            found = found and word in narrative
        shown = f"{word} + {value:,.2f}" if word else f"{value:,.2f}" if value >= 1000 else f"{value}"
        print(f"  [{'PASS' if found else 'FAIL'}] {label}: {shown}")
        all_passed = all_passed and found
    return all_passed


def find_unverified_numbers(narrative, findings):
    """Extra safety check: list numbers in the narrative that do not appear anywhere in findings.json.
    (Warning only - a number like a section count would also show up here.)"""
    allowed = set()

    def collect(item):
        if isinstance(item, dict):
            for key, value in item.items():
                collect(key)
                collect(value)
        elif isinstance(item, list):
            for value in item:
                collect(value)
        elif isinstance(item, bool):
            return
        elif isinstance(item, (int, float)):
            allowed.add(round(float(item), 2))
        elif isinstance(item, str):
            for n in re.findall(r"\d+(?:\.\d+)?", item):
                allowed.add(round(float(n), 2))

    collect(findings)
    numbers = re.findall(r"\d[\d,]*(?:\.\d+)?", narrative)
    unverified = []
    for n in numbers:
        value = round(float(n.replace(",", "")), 2)
        if value not in allowed and n not in unverified:
            unverified.append(n)
    return unverified

## Step 5: Run the pipeline

In [9]:
def main():
    findings = load_findings()
    print(f"Loaded {FINDINGS_PATH} (written by {findings.get('_source', {}).get('generated_by', 'code')})")

    # Try Gemini only if a key is set; otherwise go straight to the offline path
    if os.environ.get("GEMINI_API_KEY"):
        print(f"GEMINI_API_KEY found -> calling Gemini ({MODEL_NAME})...")
        result = generate_scr_narrative(findings)
        if result["status"] == "error":
            print(f"Gemini call failed: {result['message']}")
            print("Falling back to the offline template.")
            result = generate_scr_narrative_offline(findings)
    else:
        print("No GEMINI_API_KEY set -> using the offline template (no network, no key needed).")
        result = generate_scr_narrative_offline(findings)

    print(f"\nSource: {result['source']} | tokens used: {result['tokens']}")
    print("=" * 70)
    print(result["narrative"])
    print("=" * 70)

    print("\nNumeric accuracy check:")
    passed = check_numeric_accuracy(result["narrative"], findings)
    unverified = find_unverified_numbers(result["narrative"], findings)
    if unverified:
        print(f"  [WARN] Numbers not found in findings.json: {unverified}")
    else:
        print("  [PASS] Every number in the narrative appears in findings.json")

    # The offline template must always pass (hard check)
    if result["source"] == "offline template":
        assert passed, "Offline narrative is missing a required figure"

    # Save a successful Gemini run as the sample output (the grader checks this saved file)
    if result["source"].startswith("gemini"):
        with open(SAMPLE_OUTPUT_PATH, "w") as f:
            f.write(f"# Gemini output - model: {MODEL_NAME}, temperature: 0.0, tokens: {result['tokens']}\n")
            f.write(f"# Generated from {FINDINGS_PATH} by narrator/generate_narrative.py\n\n")
            f.write(result["narrative"] + "\n")
        print(f"\nSaved the Gemini narrative to {SAMPLE_OUTPUT_PATH}")

    # Check the saved Gemini sample (if one exists) with the same checker
    if os.path.exists(SAMPLE_OUTPUT_PATH):
        print(f"\nNumeric accuracy check of the saved Gemini sample ({SAMPLE_OUTPUT_PATH}):")
        with open(SAMPLE_OUTPUT_PATH) as f:
            check_numeric_accuracy(f.read(), findings)
    else:
        print(f"\nNo {SAMPLE_OUTPUT_PATH} yet - run once with GEMINI_API_KEY set to create it.")

## Run the narrator

In [10]:
main()

Loaded narrator/findings.json (written by analysis/clean_and_eda.py)
GEMINI_API_KEY found -> calling Gemini (gemini-3.1-flash-lite)...

Source: gemini (gemini-3.1-flash-lite) | tokens used: 973
Situation:
Our recent financial audit of sales performance reveals a raw total revenue of Rs 99,860.20. After removing duplicate orders, our cleaned total revenue stands at Rs 97,358.30. While January 2026 initially appeared to be our strongest month at Rs 29,582.10, this figure was inflated by bulk outlier orders. Once corrected, the true revenue for January 2026 is Rs 11,637.10, confirming that March 2026 remains our true peak month with Rs 20,318.90 in revenue.

Complication:
We are currently facing significant revenue leakage and operational inefficiencies. Duplicate, double-submitted orders accounted for a loss of Rs 2,501.90. Furthermore, our return rates vary drastically by payment method, with COD at 44.4%, UPI at 18.9%, and CARD at 14.7%. The most critical concern is our highest-risk se

## Download the Gemini sample output
Upload it to your repo as **narrator/sample_output.txt**.

In [11]:
if os.path.exists("narrator/sample_output.txt"):
    try:
        from google.colab import files
        files.download("narrator/sample_output.txt")
    except ImportError:
        print("Saved narrator/sample_output.txt")
else:
    print("No Gemini sample yet (no key, or the Gemini call failed) - only the offline narrative was produced.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>